# 🧠 Ridge Regression (L2 Regularization) Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **Ridge Regression**! In this notebook, we will:
1. Generate a small, noisy dataset and fit a high-degree polynomial.
2. Observe how Ordinary Least Squares (OLS) suffers from severe overfitting (high variance) and large coefficient values.
3. Apply **Ridge Regression (L2 Regularization)** to penalize large weights.
4. Visualize how varying the regularization strength ($\lambda$) smooths the fit and prevents overfitting.
5. Implement Ridge Regression from scratch using the modified **Normal Equation**:
   $$\mathbf{w} = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I}')^{-1} \mathbf{X}^T \mathbf{y}$$
   (where $\mathbf{I}'$ is the identity matrix with $I'_{0,0} = 0$ so we do not penalize the intercept).

Let's begin by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Set seed for reproducibility
np.random.seed(10)

## 1. Data Generation

We will generate only **15 training data points** using a cosine wave:
$$y = \cos(1.5 \pi x) + \epsilon$$

Fitting a high-degree polynomial (e.g., Degree 10) to only 15 points will cause OLS to overfit dramatically.

In [ ]:
# Generate training data (15 points)
X_train = np.sort(np.random.rand(15, 1) * 2 - 1, axis=0)
y_train = np.cos(1.5 * np.pi * X_train) + np.random.randn(15, 1) * 0.15

# Generate test data (30 points) for evaluation
X_test = np.sort(np.random.rand(30, 1) * 2 - 1, axis=0)
y_test = np.cos(1.5 * np.pi * X_test) + np.random.randn(30, 1) * 0.15

# Generate fine grid for plotting smooth lines
X_grid = np.linspace(-1.1, 1.1, 200).reshape(-1, 1)

# Plot training and test data
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Train Data (15 points)')
plt.scatter(X_test, y_test, color='red', alpha=0.5, label='Test Data')
plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True function')
plt.title('Small Noisy Dataset for Overfitting Demonstration')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Overfitting with OLS Polynomial Regression

Let's see what happens when we fit a Degree 10 polynomial to our 15 training points using standard Ordinary Least Squares (OLS) Linear Regression.

In [ ]:
# Transform features to Degree 10
degree = 10
poly = PolynomialFeatures(degree=degree, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_grid_poly = poly.transform(X_grid)

# Fit OLS model
ols_model = LinearRegression()
ols_model.fit(X_train_poly, y_train)

# Predict
y_grid_pred_ols = ols_model.predict(X_grid_poly)

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Train Data')
plt.plot(X_grid, y_grid_pred_ols, color='red', linewidth=2.5, label='OLS Fit (Degree 10)')
plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True Function')
plt.ylim(-2.5, 2.5)
plt.title('OLS Overfitting: Massive Oscillations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Print learned coefficients
print("Learned coefficients for OLS:")
print("Intercept:", ols_model.intercept_[0])
for idx, w in enumerate(ols_model.coef_[0]):
    print(f"w_{idx+1} (X^{idx+1}): {w:.4f}")

## 3. Regularization using Ridge Regression (L2 Penalty)

Observe how OLS coefficients are extremely large (some in the thousands or millions). This is a hallmark of overfitting.
Now, we will fit the same Degree 10 polynomial using **Ridge Regression** with different regularization strengths $\alpha$ (which represents $\lambda$).

In [ ]:
# Ridge models with different alpha strengths
alphas = [1e-5, 0.01, 1.0, 100.0]

plt.figure(figsize=(12, 6))
plt.scatter(X_train, y_train, color='blue', label='Train Data')

for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_train_poly, y_train)
    
    y_grid_pred = ridge_model.predict(X_grid_poly)
    
    # Calculate sum of squared weights
    w_squared_sum = np.sum(ridge_model.coef_ ** 2)
    
    plt.plot(X_grid, y_grid_pred, label=f'Ridge (α={alpha}, ∑w²={w_squared_sum:.2f})', linewidth=2)

plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True Function')
plt.ylim(-1.5, 1.5)
plt.title('Ridge Regression: Shrunk Coefficients Prevent Overfitting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Ridge Regression from Scratch (Normal Equation with L2 Penalty)

The Normal Equation for Ridge Regression is:
$$\mathbf{w} = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I}')^{-1} \mathbf{X}^T \mathbf{y}$$

Where:
*   $\lambda$ is the penalty parameter.
*   $\mathbf{I}'$ is a modified identity matrix. Since we do not penalize the intercept (bias) term, the top-left element is set to 0:
    $$\mathbf{I}' = \begin{bmatrix} 0 & 0 & 0 & \dots \\ 0 & 1 & 0 & \dots \\ 0 & 0 & 1 & \dots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}$$

Let's implement this feature expansion and optimization in NumPy.

In [ ]:
def get_polynomial_features(X, degree):
    m = len(X)
    X_poly = np.ones((m, 1))
    for power in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X ** power))
    return X_poly

def solve_ridge_normal_equation(X, y, lmbda):
    """
    Solve w = (X.T @ X + lmbda * I')^-1 @ X.T @ y
    """
    n_features = X.shape[1]
    
    # Create modified identity matrix (first element is 0 for bias)
    I_prime = np.identity(n_features)
    I_prime[0, 0] = 0.0
    
    # Compute formula components
    left_side = X.T @ X + lmbda * I_prime
    left_side_inv = np.linalg.inv(left_side)
    w = left_side_inv @ X.T @ y
    return w

# Transform X_train to Degree 10 with bias included in our function
X_train_scratch = get_polynomial_features(X_train, degree=10)
X_grid_scratch = get_polynomial_features(X_grid, degree=10)

# Set lambda
lmbda = 0.01

# Solve for weights
weights_scratch = solve_ridge_normal_equation(X_train_scratch, y_train, lmbda)

# Predict
y_grid_pred_scratch = X_grid_scratch @ weights_scratch

# Evaluate on Train vs Test
X_test_scratch = get_polynomial_features(X_test, degree=10)
y_train_pred = X_train_scratch @ weights_scratch
y_test_pred = X_test_scratch @ weights_scratch

print(f"Scratch Ridge Model (λ={lmbda}) Performance:")
print(f"Train MSE: {mean_squared_error(y_train, y_train_pred):.4f}")
print(f"Test MSE:  {mean_squared_error(y_test, y_test_pred):.4f}")

# Plot scratch fit
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Train Data')
plt.plot(X_grid, y_grid_pred_scratch, color='green', linewidth=2.5, label=f'Scratch Ridge (λ={lmbda})')
plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True Function')
plt.ylim(-1.5, 1.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 💡 Connection to Deep Learning & YOLO
*   **Weight Decay:** In deep learning optimization (such as training YOLO), L2 regularization is called **Weight Decay**. By adding a small fraction of the weights to the gradients at each step, the optimizer prevents the network from learning extremely large weight parameters. This forces the model to learn simple, smooth feature detectors rather than focusing too intensely on specific pixel locations or patterns, which directly reduces overfitting and improves object classification accuracy in noisy environments.